# Adult Income Dataset — Data Cleaning & Preprocessing

## 1. Introduction

This notebook focuses on cleaning and preparing the Adult Census Income dataset for future machine learning tasks.

Building on the findings from the EDA and Statistical Analysis notebook, this stage will address data quality issues, handle missing values, investigate duplicates, and prepare features through justified preprocessing decisions.

The original dataset will be preserved to ensure reproducibility and prevent unintended data modifications.

**Note:** This notebook covers data cleaning and preprocessing only. Model training and evaluation are outside its scope.

In [95]:
pip install ucimlrepo

In [96]:
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo
# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [97]:
# Load the Original Dataset

adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.copy()

# Preserve the original dataset without modifications
df_raw = X.copy()
df_raw["income"] = y.iloc[:, 0].copy()

print(f"Dataset Shape: {df_raw.shape}")
display(df_raw.head())

Dataset Shape: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [98]:
print("Dataset Information:")
df_raw.info()

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


In [99]:
print("\nMissing Values (NaN):")
display(df_raw.isna().sum()[df_raw.isna().sum() > 0])

print("\nUnknown Markers ('?'):")
unknown_counts = df_raw.select_dtypes(include="object").eq("?").sum()
display(unknown_counts[unknown_counts > 0])


Missing Values (NaN):


,0
workclass,963
occupation,966
native-country,274



Unknown Markers ('?'):


,0
workclass,1836
occupation,1843
native-country,583


In [100]:
print("\nDuplicate Rows:")
print(df_raw.duplicated().sum())


Duplicate Rows:
29


In [101]:
print("\nOriginal Target Distribution:")
display(df_raw["income"].value_counts(dropna=False))


Original Target Distribution:


,count
income,
<=50K,24720
<=50K.,12435
>50K,7841
>50K.,3846


## 2. Missing and Unknown Values Investigation

The initial inspection revealed that missing categorical values are represented in two different forms: actual `NaN` values and the literal `"?"` marker.

Before applying any cleaning strategy, we will investigate their distribution and overlap to understand the extent of missing information in each affected feature.

In [102]:
affected_cols = ["workclass", "occupation", "native-country"]

missing_analysis = pd.DataFrame({
    "NaN Count": df_raw[affected_cols].isna().sum(),
    "? Count": df_raw[affected_cols].eq("?").sum(),
})

missing_analysis["Total Unknown"] = (
    missing_analysis["NaN Count"] + missing_analysis["? Count"]
)

missing_analysis["Unknown Percentage"] = (
    missing_analysis["Total Unknown"] / len(df_raw) * 100
)

display(missing_analysis.round(2))

,NaN Count,? Count,Total Unknown,Unknown Percentage
workclass,963,1836,2799,5.73
occupation,966,1843,2809,5.75
native-country,274,583,857,1.75


In [103]:
# Check whether NaN and '?' occur in the same rows

for col in affected_cols:
    nan_mask = df_raw[col].isna()
    unknown_mask = df_raw[col].eq("?")

    print(f"\n{col}")
    print(f"NaN only: {(nan_mask & ~unknown_mask).sum()}")
    print(f"'?' only: {(unknown_mask & ~nan_mask).sum()}")
    print(f"Both: {(nan_mask & unknown_mask).sum()}")


workclass
NaN only: 963
'?' only: 1836
Both: 0

occupation
NaN only: 966
'?' only: 1843
Both: 0

native-country
NaN only: 274
'?' only: 583
Both: 0


## 3. Standardizing Missing Value Representation

The investigation confirmed that actual `NaN` values and the `"?"` marker occur in separate rows.

To ensure consistent missing-value handling, all `"?"` markers in the affected categorical features will be converted to `NaN`.

The original dataset (`df_raw`) will remain unchanged, while all subsequent cleaning operations will be performed on a separate working copy.

In [104]:
df = df_raw.copy()

# Convert unknown markers into NaN
affected_cols = ["workclass", "occupation", "native-country"]

df[affected_cols] = df[affected_cols].replace("?", np.nan)

print("Missing Values After Standardization:")
display(df[affected_cols].isna().sum())

print("\nRemaining '?' Markers:")
display(df[affected_cols].eq("?").sum())

Missing Values After Standardization:


,0
workclass,2799
occupation,2809
native-country,857



Remaining '?' Markers:


,0
workclass,0
occupation,0
native-country,0


In [105]:
# Verify that the raw dataset remains unchanged

print("Original Dataset:")
display(df_raw[affected_cols].isna().sum())

print("\nCleaned Working Copy:")
display(df[affected_cols].isna().sum())

print("\nRaw Dataset Shape:", df_raw.shape)
print("Working Copy Shape:", df.shape)

Original Dataset:


,0
workclass,963
occupation,966
native-country,274



Cleaned Working Copy:


,0
workclass,2799
occupation,2809
native-country,857



Raw Dataset Shape: (48842, 15)
Working Copy Shape: (48842, 15)


## 4. Duplicate Rows Investigation

Duplicate records can affect data quality and may lead to unintended overrepresentation of certain observations.

Before removing any duplicates, we will inspect the duplicated records and investigate their characteristics to determine whether removal is justified.

In [106]:
# Verify exact duplicate records

print("Total duplicated rows:", df.duplicated().sum())

print(
    "Duplicated rows with different target labels:",
    df.groupby(list(df.columns[:-1]), dropna=False)["income"]
      .nunique()
      .gt(1)
      .sum()
)

Total duplicated rows: 29
Duplicated rows with different target labels: 28


## 5. Target Label Standardization

The original dataset contains four target labels due to differences in the representation of the training and test subsets.

The trailing period does not indicate a different income class. Therefore, a standardized target column will be created by removing the trailing period while preserving the original target labels.

In [107]:
# Standardize the target labels without modifying the original column

df["income_clean"] = df["income"].str.strip().str.rstrip(".")

print("Original Target Distribution:")
display(df["income"].value_counts())

print("\nStandardized Target Distribution:")
display(df["income_clean"].value_counts())

Original Target Distribution:


,count
income,
<=50K,24720
<=50K.,12435
>50K,7841
>50K.,3846



Standardized Target Distribution:


,count
income_clean,
<=50K,37155
>50K,11687


In [108]:
# Check exact duplicates after target standardization

feature_cols = df.columns.drop(["income", "income_clean"])

print("Duplicate rows using original target:",
      df.duplicated(subset=list(df.columns.drop("income_clean"))).sum())

print("Duplicate rows including standardized target:",
      df.duplicated(subset=list(feature_cols) + ["income_clean"]).sum())

Duplicate rows using original target: 29
Duplicate rows including standardized target: 52


In [109]:
# Check whether identical features still have conflicting standardized targets

conflicts = (
    df.groupby(list(feature_cols), dropna=False)["income_clean"]
      .nunique()
      .gt(1)
)

print("Conflicting feature combinations after target standardization:",
      conflicts.sum())

Conflicting feature combinations after target standardization: 5


In [110]:
# Inspect conflicting records after target standardization

conflicting_features = (
    df.groupby(list(feature_cols), dropna=False)["income_clean"]
      .nunique()
)

conflicting_features = conflicting_features[
    conflicting_features > 1
].index

conflict_records = (
    df.set_index(list(feature_cols))
      .loc[conflicting_features]
      .reset_index()
      .sort_values(list(feature_cols))
)

display(
    conflict_records[
        list(feature_cols) + ["income", "income_clean"]
    ]
)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income,income_clean
0,34,Private,169583,Masters,14,Never-married,Exec-managerial,Not-in-family,White,Female,0,0,50,United-States,<=50K,<=50K
1,34,Private,169583,Masters,14,Never-married,Exec-managerial,Not-in-family,White,Female,0,0,50,United-States,>50K.,>50K
2,39,Private,138192,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K,<=50K
3,39,Private,138192,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,>50K,>50K
4,41,Private,242804,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,40,United-States,<=50K.,<=50K
5,41,Private,242804,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,40,United-States,>50K.,>50K
6,46,Private,224559,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,Black,Male,0,0,40,United-States,<=50K,<=50K
7,46,Private,224559,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,Black,Male,0,0,40,United-States,>50K.,>50K
8,46,Private,235646,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,>50K,>50K
9,46,Private,235646,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K.,<=50K


In [111]:
# Summarize the conflicting target labels

conflict_summary = (
    conflict_records.groupby(list(feature_cols), dropna=False)
    ["income_clean"]
    .value_counts()
    .unstack(fill_value=0)
)

display(conflict_summary)

income_clean                                                                                                                                                         <=50K  \
age workclass fnlwgt education education-num marital-status     occupation       relationship  race  sex    capital-gain capital-loss hours-per-week native-country          
34  Private   169583 Masters   14            Never-married      Exec-managerial  Not-in-family White Female 0            0            50             United-States       1   
39  Private   138192 Bachelors 13            Married-civ-spouse Craft-repair     Husband       White Male   0            0            40             United-States       1   
41  Private   242804 HS-grad   9             Married-civ-spouse Transport-moving Husband       White Male   0            0            40             United-States       1   
46  Private   224559 HS-grad   9             Married-civ-spouse Transport-moving Husband       Black Male   0            0            40             United-States       1   
              235646 HS-grad   9             Married-civ-spouse Craft-repair     Husband       White Male   0            0            40             United-States       1   

income_clean                                                                                                                                                         >50K  
age workclass fnlwgt education education-num marital-status     occupation       relationship  race  sex    capital-gain capital-loss hours-per-week native-country        
34  Private   169583 Masters   14            Never-married      Exec-managerial  Not-in-family White Female 0            0            50             United-States      1  
39  Private   138192 Bachelors 13            Married-civ-spouse Craft-repair     Husband       White Male   0            0            40             United-States      1  
41  Private   242804 HS-grad   9             Married-civ-spouse Transport-moving Husband       White Male   0            0            40             United-States      1  
46  Private   224559 HS-grad   9             Married-civ-spouse Transport-moving Husband       Black Male   0            0            40             United-States      1  
              235646 HS-grad   9             Married-civ-spouse Craft-repair     Husband       White Male   0            0            40             United-States      1

In [112]:
# Define feature columns explicitly

feature_cols = [
    col for col in df.columns
    if col not in ["income", "income_clean"]
]

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 14
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']


In [113]:
df = df.drop_duplicates(
    subset=feature_cols + ["income_clean"]
).copy()

print("Dataset shape after duplicate removal:", df.shape)

Dataset shape after duplicate removal: (48790, 16)


In [114]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": df.isna().mean() * 100
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing Percentage", ascending=False)

display(missing_summary.round(2))

,Missing Count,Missing Percentage
occupation,2805,5.75
workclass,2795,5.73
native-country,856,1.75


In [115]:
missing_patterns = pd.crosstab(
    df["workclass"].isna(),
    df["occupation"].isna(),
    margins=True
)

display(missing_patterns)

occupation,False,True,All
workclass,,,
False,45985,10,45995
True,0,2795,2795
All,45985,2805,48790


In [116]:
print(
    "Both missing:",
    (df["workclass"].isna() & df["occupation"].isna()).sum()
)

print(
    "Workclass only:",
    (df["workclass"].isna() & df["occupation"].notna()).sum()
)

print(
    "Occupation only:",
    (df["workclass"].notna() & df["occupation"].isna()).sum()
)

Both missing: 2795
Workclass only: 0
Occupation only: 10


In [117]:
categorical_cols = [
    "workclass",
    "occupation",
    "native-country"
]

for col in categorical_cols:

    income_rate = (
        df.groupby(col, dropna=False)["income_clean"]
          .apply(lambda x: (x == ">50K").mean() * 100)
          .sort_values(ascending=False)
    )

    print(f"\nHigh Income Rate by {col} (%)")
    display(income_rate.round(2).to_frame("High Income Rate (%)"))


High Income Rate by workclass (%)


,High Income Rate (%)
workclass,
Self-emp-inc,55.37
Federal-gov,39.18
Local-gov,29.56
Self-emp-not-inc,27.89
State-gov,26.75
Private,21.80
Without-pay,9.52
NaN,9.48
Never-worked,0.00



High Income Rate by occupation (%)


,High Income Rate (%)
occupation,
Exec-managerial,47.78
Prof-specialty,45.13
Armed-Forces,33.33
Protective-serv,31.36
Tech-support,29.07
Sales,26.81
Craft-repair,22.65
Transport-moving,20.42
Adm-clerical,13.70



High Income Rate by native-country (%)


,High Income Rate (%)
native-country,
France,42.11
India,41.06
Taiwan,40.00
Iran,37.29
England,37.01
Greece,36.73
Japan,34.78
Yugoslavia,34.78
Canada,34.62


In [118]:
from scipy.stats import chi2_contingency

for col in categorical_cols:

    contingency_table = pd.crosstab(
        df[col].fillna("Missing"),
        df["income_clean"]
    )

    chi2, p_value, dof, expected = chi2_contingency(
        contingency_table
    )

    print(f"\n{col}")
    print(f"Chi-square statistic: {chi2:.2f}")
    print(f"P-value: {p_value:.6g}")


workclass
Chi-square statistic: 1609.50
P-value: 0

occupation
Chi-square statistic: 5975.47
P-value: 0

native-country
Chi-square statistic: 449.41
P-value: 3.75164e-70


features are related with the target so we cannot drop their coulmns

In [119]:

# Apply mode imputation to categorical features

missing_cols = ["workclass", "occupation", "native-country"]

for col in missing_cols:
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)

    print(f"{col}: Missing values replaced with '{mode_value}'")


workclass: Missing values replaced with 'Private'
occupation: Missing values replaced with 'Prof-specialty'
native-country: Missing values replaced with 'United-States'


In [120]:
df[missing_cols].isna().sum()

,0
workclass,0
occupation,0
native-country,0


In [121]:
# Inspect dataset data types

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48790 entries, 0 to 48841
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48790 non-null  int64 
 1   workclass       48790 non-null  object
 2   fnlwgt          48790 non-null  int64 
 3   education       48790 non-null  object
 4   education-num   48790 non-null  int64 
 5   marital-status  48790 non-null  object
 6   occupation      48790 non-null  object
 7   relationship    48790 non-null  object
 8   race            48790 non-null  object
 9   sex             48790 non-null  object
 10  capital-gain    48790 non-null  int64 
 11  capital-loss    48790 non-null  int64 
 12  hours-per-week  48790 non-null  int64 
 13  native-country  48790 non-null  object
 14  income          48790 non-null  object
 15  income_clean    48790 non-null  object
dtypes: int64(6), object(10)
memory usage: 6.3+ MB


In [123]:
df = df.drop_duplicates(
    subset=feature_cols + ["income_clean"]
).copy()

print("Final shape:", df.shape)
print("Remaining duplicates:", df.duplicated(
    subset=feature_cols + ["income_clean"]
).sum())

Final shape: (48789, 16)
Remaining duplicates: 0


In [124]:

# Final dataset validation

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nTarget Distribution:")
display(
    df["income_clean"]
    .value_counts()
    .to_frame("Count")
)

print("\nTarget Distribution (%):")
display(
    df["income_clean"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("Percentage")
)

print("\nUnique Target Values:")
print(df["income_clean"].unique())


Dataset Shape: (48789, 16)

Column Names:
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income', 'income_clean']

Target Distribution:


,Count
income_clean,
<=50K,37108
>50K,11681



Target Distribution (%):


,Percentage
income_clean,
<=50K,76.06
>50K,23.94



Unique Target Values:
['<=50K' '>50K']


In [125]:

# Create the final cleaned dataset

final_df = df.drop(columns=["income"]).copy()

# Rename the standardized target column
final_df = final_df.rename(columns={"income_clean": "income"})

print("Final Dataset Shape:", final_df.shape)
display(final_df.head())


Final Dataset Shape: (48789, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [126]:
# Final integrity checks

print("Missing values:")
display(final_df.isna().sum())

print("\nTarget values:")
print(final_df["income"].value_counts(dropna=False))

print("\nData types:")
final_df.info()

Missing values:


,0
age,0
workclass,0
fnlwgt,0
education,0
education-num,0
marital-status,0
occupation,0
relationship,0
race,0
sex,0



Target values:
income
<=50K    37108
>50K     11681
Name: count, dtype: int64

Data types:
<class 'pandas.core.frame.DataFrame'>
Index: 48789 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48789 non-null  int64 
 1   workclass       48789 non-null  object
 2   fnlwgt          48789 non-null  int64 
 3   education       48789 non-null  object
 4   education-num   48789 non-null  int64 
 5   marital-status  48789 non-null  object
 6   occupation      48789 non-null  object
 7   relationship    48789 non-null  object
 8   race            48789 non-null  object
 9   sex             48789 non-null  object
 10  capital-gain    48789 non-null  int64 
 11  capital-loss    48789 non-null  int64 
 12  hours-per-week  48789 non-null  int64 
 13  native-country  48789 non-null  object
 14  income          48789 non-null  object
dtypes: int64(6), object(9)
memory usage: 6.0+ MB


In [127]:
# Export the final cleaned dataset

output_path = "adult_income_cleaned.csv"

final_df.to_csv(output_path, index=False)

print(f"Dataset successfully exported to: {output_path}")

Dataset successfully exported to: adult_income_cleaned.csv


In [128]:
from google.colab import files

files.download("adult_income_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

In this notebook, the Adult Census Income dataset underwent a structured data cleaning and preprocessing workflow.

The main steps included:

- Standardizing inconsistent target labels into two classes: `<=50K` and `>50K`.
- Investigating and removing exact duplicate records while preserving records with conflicting target labels.
- Analyzing missing values and applying mode imputation to the affected categorical features.
- Validating data types and checking the integrity of the final dataset.

The resulting dataset contains **48,789 records and 15 columns**, including 14 features and one standardized target column (`income`).

The final dataset contains no missing values or remaining exact duplicates under the selected duplicate definition and is ready for future analytical and machine learning workflows.

## Limitations & Considerations

Despite completing the cleaning and preprocessing workflow, several limitations should be considered:

1. **Mode Imputation:** Replacing missing categorical values with the most frequent category may distort the original category distribution and reduce variability. The imputed values should not be interpreted as confirmed original observations.

2. **Duplicate Handling:** Duplicate records were identified using the feature values and standardized target. Records sharing identical features but having different target labels were retained because their target values conflict.

3. **Target Imbalance:** The target variable is imbalanced, with the `<=50K` class representing 76.06% of observations and the `>50K` class representing 23.94%.

4. **Statistical Associations:** Observed relationships between features and income do not imply causation.

5. **Future Machine Learning:** Any learned preprocessing parameters, such as imputation values, should be fitted using training data only when building predictive models to avoid data leakage.

## Final Outcome

The cleaned dataset was exported as `adult_income_cleaned.csv`.

| Metric | Result |
|---|---:|
| Original Records | 48,842 |
| Final Records | 48,789 |
| Features | 14 |
| Target Classes | 2 |
| Final Columns | 15 |
| Missing Values | 0 |
| Remaining Exact Duplicates | 0 |

The prepared dataset provides a consistent foundation for future machine learning experiments and further analysis.